# Spin-up Quick Check Notebook
This notebook is organized for spin-up diagnostics on monthly model output.

## Task 1: Domain map animation of a selected biological variable
- Input file: `dws_500m.3d.201501.nc`
- Goal: animate a selected variable over time on the model map
- Output: an animated GIF (optional) and inline animation preview

## Task 2: Aggregated time series of a selected biological variable for the whole domain
- Cell 2 (Task 2): build and plot the domain-mean trend at a selected layer (4d) or 3d variable
- Input file: `dws_500m.3d.201501.nc`
- Goal: visualize time series of spatially averaged values over the whole domain also for a specific vertical level;
- Output: a time-series line plot

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

from scipy.interpolate import RegularGridInterpolator
from pathlib import Path
from IPython.display import HTML

from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML

In [ ]:
# meta data:
#! BFM biological model

# pelagic variables (group PelVariables):
#! pelagic  (O)              O2:   Oxygen (mmol/m3)
#! pelagic  (P)              N1:   Phosphate (mmol/m3)
#! pelagic  (N)              N3:   Nitrate (mmol/m3)
#! pelagic  (N)              N4:   Ammonium (mmol/m3)
#! pelagic  (Si)             N5:   Silicate (mmol/m3)
#! pelagic  (R)              N6:   Reduction Equivalents (mmol/m3)
#! pelagic  (N)              O4:   N2-sink (mmol/m3)
#! pelagic  (CNP)            B1:   Pelagic Bacteria
#! pelagic  (CNPSiI)         P1:   Diatoms (group PhytoPlankton))
#! pelagic  (CNPSiI)         P2:   Flagellates (group PhytoPlankton))
#! pelagic  (CNPSiI)         P3:   PicoPhytoPlankton (group PhytoPlankton))
#! pelagic  (CNPSiI)         P4:   Dinoflagellates (group PhytoPlankton))
#! pelagic  (CNP)            Z3:   Carnivorous mesozooplankton (group MesoZooPlankton))
#! pelagic  (CNP)            Z4:   Omnivorous mesozooplankton (group MesoZooPlankton))
#! pelagic  (CNP)            Z5:   Microzooplankton (group MicroZooPlankton))
#! pelagic  (CNP)            Z6:   Heterotrophic nanoflagellates (HNAN) (group MicroZooPlankton))
#! pelagic  (CNPSi)          R1:   Labile Organic Carbon (LOC)
#! pelagic  (C)              R2:   CarboHydrates (sugars)
#! pelagic  (CNPSi)          R6:   Particulate Organic Carbon (POC)
#! pelagic  (C)              R7:   Refractory Disoolved Organic Carbon

# Benthic variables (group BenVariables):
# ! benthic  (CNP)            Y1:   Epibenthos (group BenOrganisms))
# ! benthic  (CNP)            Y2:   Deposit feeders (group BenOrganisms))
# ! benthic  (CNP)            Y3:   Suspension feeders (group BenOrganisms))
# ! benthic  (CNP)            Y4:   Meiobenthos (group ! BenOrganisms))
# ! benthic  (CNP)            Y5:   Benthic predators (group BenOrganisms))
# ! benthic  (CNPSi)          Q1:   Labile organic carbon (group BenDetritus))
# ! benthic  (CNPSi)          Q11:  Labile organic carbon (group BenDetritus))
# ! benthic  (CNPSi)          Q6:   Particulate organic carbon (group BenDetritus))
# ! benthic  (CNP)            H1:   Aerobic benthic bacteria (group BenBacteria))
# ! benthic  (CNP)            H2:   Anaerobic benthic bacteria (group BenBacteria))
# ! benthic  (P)              K1:   Phosphate in oxic layer (group BenthicPhosphate))
# ! benthic  (P)              K11:  Phosphate in denit layer (group BenthicPhosphate))
# ! benthic  (P)              K21:  Phosphate in anoxic layer (group BenthicPhosphate))
# ! benthic  (N)              K4:   Ammonium in oxic layer (group BenthicAmmonium))
# ! benthic  (N)              K14:  Ammonium in denit layer (group BenthicAmmonium))
# ! benthic  (N)              K24:  Ammonium in anoxic layer (group BenthicAmmonium))
# ! benthic  (R)              K6:   Reduction equivalents 
# ! benthic  (M)              D1:   Oxygen penetration depth
# ! benthic  (M)              D2:   Denitrification depth 
# ! benthic  (M)              D6:   Depth distribution factor organic C 
# ! benthic  (M)              D7:   Depth distribution factor organic N
# ! benthic  (M)              D8:   Depth distribution factor organic P
# ! benthic  (M)              D9:   Depth distribution factor organic Si
# ! benthic  (O)              G2:   Benthic O2


# List of variables for analysis and visualization
vars_list = [
    'elev',
    'temp',
    'salt',
    'O2o', 
    'netPPm2',
          'N1p',
          'N3n',
          'N4n',
          'N5s',
          'N6r',
#          'B1c',
#          'Bac',
          'P1c',
          'P2c',
          'P3c',
          'P4c',
          'P5c',
          'P6c',
#	  'P1l',
#          'P2l',
#          'P3l',
#          'P4l',
 #         'P5l',
 #         'P6l',
 #         'Z2c',
 #         'Z3c',
 #         'Z4c',
 #         'Z5c',
 #         'Z6c',
 #         'R1c',
 #         'R2c',
 #         'R3c',
 #         'R6c',
 #         'RZc',
 #         'Q1c',
 #         'Q11c',
 #         'Q6c',
          'Chla',
#          'H1c',
#          'H2c',
#          'HNc',
#          'Hac',
          'Y1c',
          'Y2c',
          'Y3c',
#          'Y4c',
          'Y5c',
          'Yy3c',
#          'K6r',
#          'K16r',
#          'K26r',
#          'K5s',
#          'K15s',
#          'K3n',
#          'K4n',
#         'K13n',
#          'K14n',
#          'K24n',
#          'K1p',
#          'K11p',
#         'K21p',
#          'D1m',
#          'D2m',
#          'O3c',
#          'pCO2',
#          'CO2',
#          'HCO3',
#          'CO3',
#          'pH',
#          'Ac'
#          'G3h'
#          'G13h'
#          'G23h'
#          'G3c'
#          'G13c'
#          'G23c'
#          'G14n'
#          'Acae'
#          'Acan'
#          'DICae'
#          'DICan'
#          'pHae'
#          'pHan'
#          'pCO2ae'
#          'pCO2an'
#          'G3h'
#          'G13h'
#          'BP1c'
#          'ETW',
#          'irrenh',
#          'turenh',      
]

topo_file = Path('/export/lv9/projects/dws/model_input/bathymetry/topo_dws_500m.nc')
topo_ds = xr.open_dataset(topo_file)
xc = topo_ds['xc'].values
yc = topo_ds['yc'].values

DATA_DIR = Path('/export/lv9/projects/dws/model_output/archived_runs/spinup_02/')
FILE_PATTERN = 'dws_500m.3d.2015??.nc'

# Crop bounds from topo file
XC_MIN, XC_MAX = float(xc.min()), float(xc.max())
YC_MIN, YC_MAX = float(yc.min()), float(yc.max())
print(f'Topo-derived crop bounds — xc: {XC_MIN:.0f}–{XC_MAX:.0f}, yc: {YC_MIN:.0f}–{YC_MAX:.0f}')

files = sorted(DATA_DIR.glob(FILE_PATTERN))
if not files:
    raise FileNotFoundError(f'No files found: {FILE_PATTERN} in {DATA_DIR}')



In [ ]:
# Merge monthly NetCDF files into a single file for the year 2015 (run once)
NETCDF_PATH = Path('/export/lv9/user/qzhan/archived_output/hydro_bio/spinup/2015_run_01/')
for month in range(1, 13):
    month_dir = NETCDF_PATH / f"{month:02d}"
    files = sorted(month_dir.glob("dws_500m.3d.00??.nc"))
    if files:
        ds = xr.open_mfdataset(files, combine='by_coords')
        out_path = NETCDF_PATH / f"dws_500m.3d.2015{month:02d}.nc"
        ds.to_netcdf(out_path)
        ds.close()

In [ ]:
# Time series trends for all variables in vars_list over the full year 2015
DATA_DIR = Path('/export/lv9/projects/dws/model_output/archived_runs/spinup_05/')
#DATA_DIR = Path('/export/lv9/user/qzhan/archived_output/hydro_bio/spinup/2015_run_01/')
#DATA_DIR = Path('/export/lv9/user/qzhan/archived_output/hydro_bio/spinup/2015_run_02/')
FILE_PATTERN = 'dws_500m.3d.2015??.nc'
SURFACE_LAYER_INDEX = 10   # Use top 11; surface layer by default
USE_DAILY_MEAN = False     # Set True if you want daily-mean smoothing

files = sorted(DATA_DIR.glob(FILE_PATTERN))
if not files:
    raise FileNotFoundError(f'No files found with pattern: {FILE_PATTERN} in {DATA_DIR}')

# Concatenate by file order to avoid alignment failures when duplicate timestamps exist across files.
ds = xr.open_mfdataset(
    files,
    combine='nested',
    concat_dim='time',
    decode_times=True,
    data_vars='minimal',
    coords='minimal',
    compat='override',
    join='override',
)

def _find_time_dim(da: xr.DataArray) -> str:
    for d in da.dims:
        if 'time' in d.lower():
            return d
    raise ValueError(f'No time dimension found in {da.dims}')

def _find_vertical_dim(da: xr.DataArray, time_dim: str) -> str | None:
    candidates = ('level', 'z', 'sigma', 'layer', 'lev', 'depth', 'nmesh2_layer_3d')
    for d in da.dims:
        if d != time_dim and any(k in d.lower() for k in candidates):
            return d
    return None

def _drop_duplicate_time(da: xr.DataArray, time_dim: str) -> xr.DataArray:
    # Keep first occurrence when duplicate time stamps are present.
    time_values = np.asarray(da[time_dim].values)
    _, keep_idx = np.unique(time_values, return_index=True)
    keep_idx = np.sort(keep_idx)
    if keep_idx.size < time_values.size:
        da = da.isel({time_dim: keep_idx})
    return da

nvars = len(vars_list)
ncols = 4
nrows = int(np.ceil(nvars / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 2.8 * nrows), sharex=True)
axes = np.atleast_1d(axes).ravel()

missing_vars = []
failed_vars = []

for i, vname in enumerate(vars_list):
    ax = axes[i]
    try:
        if vname not in ds.variables:
            missing_vars.append(vname)
            ax.text(0.5, 0.5, f'{vname}\nnot found', ha='center', va='center', transform=ax.transAxes)
            ax.set_title(vname)
            ax.set_axis_off()
            continue

        da = ds[vname].squeeze(drop=True)
        time_dim = _find_time_dim(da)
        z_dim = _find_vertical_dim(da, time_dim)

        if z_dim is not None:
            da_use = da.isel({z_dim: SURFACE_LAYER_INDEX})
        else:
            da_use = da

        da_use = _drop_duplicate_time(da_use, time_dim)

        spatial_dims = [d for d in da_use.dims if d != time_dim]
        if not spatial_dims:
            raise ValueError('No spatial dimensions left for domain averaging.')
        
        if vname.lower() == 'chla' or vname.lower() == 'netppm2':
            da_use = da_use.where(da_use >= 0)  # Mask out negative Chla values before averaging
        series = da_use.mean(dim=spatial_dims, skipna=True)

        if USE_DAILY_MEAN:
            series = series.resample({time_dim: '1D'}).mean(skipna=True)

        ax.plot(series[time_dim].values, series.values, lw=1.6)
        units = da.attrs.get('units', '')
        ylabel = f'{vname} [{units}]' if units else vname
        ax.set_title(vname)
        ax.set_ylabel(ylabel, fontsize=8)
        ax.grid(True, alpha=0.3)

    except Exception as e:
        failed_vars.append((vname, str(e)))
        ax.text(0.5, 0.5, f'{vname}\nfailed', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(vname)
        ax.set_axis_off()

for j in range(nvars, len(axes)):
    axes[j].set_visible(False)

for ax in axes[:nvars]:
    ax.tick_params(axis='x', labelrotation=35)

fig.suptitle('2015 Domain-Mean Trends for Selected Variables', y=1.02)
plt.tight_layout()
plt.show()

if missing_vars:
    print('Variables not found in dataset:')
    print(missing_vars)

if failed_vars:
    print('Variables that failed during processing:')
    for name, err in failed_vars:
        print(f'- {name}: {err}')

In [ ]:
# ------------------------------ Cell 6: elevation animation for full year ------------------------------
# Goal: animate elevation over the whole domain for all of 2015.
# Supports both 3D (time, y, x) and 4D (time, z, y, x) variables.

DATA_DIR = Path('/export/lv9/projects/dws/model_output/archived_runs/spinup_05/')
FILE_PATTERN = 'dws_500m.3d.2015??.nc'
PLOTVAR = 'elev'
LAYER_INDEX = 10  # used only if a vertical dimension exists
TIME_STEP = 1
CMAP = 'viridis'

# Derive colour range from the actual data
#VMIN = float(field.min())
#VMAX = float(field.max())
#print(f"Color range: {VMIN:.4f} - {VMAX:.4f}" + (" " + plot_units if plot_units else ""))

files = sorted(DATA_DIR.glob(FILE_PATTERN))
if not files:
    raise FileNotFoundError(f'No files found with pattern: {FILE_PATTERN} in {DATA_DIR}')

# Concatenate in file order to avoid alignment errors from duplicate timestamps.
ds = xr.open_mfdataset(
    files,
    combine='nested',
    concat_dim='time',
    decode_times=True,
    data_vars='minimal',
    coords='minimal',
    compat='override',
    join='override',
)

def _find_time_dim(da: xr.DataArray) -> str:
    for d in da.dims:
        if 'time' in d.lower():
            return d
    raise ValueError(f'No time-like dimension found in {da.dims}')

def _find_vertical_dim(da: xr.DataArray, time_dim: str) -> str | None:
    candidates = ('level', 'z', 'sigma', 'layer', 'lev', 'depth', 'nmesh2_layer_3d')
    for d in da.dims:
        if d == time_dim:
            continue
        if any(k in d.lower() for k in candidates):
            return d
    return None

def _pick_coord_name(ds: xr.Dataset, candidates: tuple[str, ...]) -> str | None:
    for name in candidates:
        if name in ds.variables:
            return name
    return None

def _to_float(values) -> np.ndarray:
    if np.ma.isMaskedArray(values):
        values = np.ma.filled(values, np.nan)
    return np.asarray(values, dtype=float)

def _drop_duplicate_time(da: xr.DataArray, time_dim: str) -> xr.DataArray:
    time_values = np.asarray(da[time_dim].values)
    _, keep_idx = np.unique(time_values, return_index=True)
    keep_idx = np.sort(keep_idx)
    if keep_idx.size < time_values.size:
        da = da.isel({time_dim: keep_idx})
    return da

if PLOTVAR not in ds.variables:
    raise KeyError(f"Variable '{PLOTVAR}' not found. Available: {sorted(ds.data_vars)}")

da = ds[PLOTVAR].squeeze(drop=True)
plot_units = da.attrs.get('units', '')
plot_units_label = f" [{plot_units}]" if plot_units else ''
time_dim = _find_time_dim(da)
z_dim = _find_vertical_dim(da, time_dim)

if z_dim is not None:
    if LAYER_INDEX >= da.sizes[z_dim] or LAYER_INDEX < -da.sizes[z_dim]:
        raise IndexError(f"LAYER_INDEX={LAYER_INDEX} out of bounds for '{z_dim}' size {da.sizes[z_dim]}")
    field = da.isel({z_dim: LAYER_INDEX})
    layer_text = f' | layer {LAYER_INDEX}'
else:
    field = da
    layer_text = ''

field = _drop_duplicate_time(field, time_dim)

# Derive colour range from the actual data
#VMIN = -1.5
#VMAX = 1.5
VMIN = float(field.min())
VMAX = float(field.max())
print(f"Color range: {VMIN:.4f} - {VMAX:.4f}" + (" " + plot_units if plot_units else ""))

xy_dims = [d for d in field.dims if d != time_dim]
if len(xy_dims) != 2:
    raise ValueError(f'Expected 2 horizontal dims after slicing, got {xy_dims} from {field.dims}')
y_dim, x_dim = xy_dims

times = field[time_dim]
frame_indices = np.arange(0, field.sizes[time_dim], TIME_STEP)
if len(frame_indices) == 0:
    raise ValueError('No frames selected. Check TIME_STEP and time length.')

ny = field.sizes[y_dim]
nx = field.sizes[x_dim]

lon_name = _pick_coord_name(ds, ('lonc', 'lon', 'longitude'))
lat_name = _pick_coord_name(ds, ('latc', 'lat', 'latitude'))

use_geo = False
if lon_name is not None and lat_name is not None:
    lon_da = ds[lon_name]
    lat_da = ds[lat_name]
    if y_dim in lon_da.dims and x_dim in lon_da.dims and y_dim in lat_da.dims and x_dim in lat_da.dims:
        x_geo = _to_float(lon_da.transpose(y_dim, x_dim).values)
        y_geo = _to_float(lat_da.transpose(y_dim, x_dim).values)
        if x_geo.shape == (ny, nx) and y_geo.shape == (ny, nx):
            if np.isfinite(x_geo).all() and np.isfinite(y_geo).all():
                x_plot, y_plot = x_geo, y_geo
                use_geo = True

if not use_geo:
    x_plot, y_plot = np.meshgrid(np.arange(nx, dtype=float), np.arange(ny, dtype=float))

fig, ax = plt.subplots(figsize=(8, 6))
plt.close(fig)

norm = plt.Normalize(vmin=VMIN, vmax=VMAX)
sm = plt.cm.ScalarMappable(norm=norm, cmap=CMAP)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, fraction=0.035, pad=0.03)
cbar.set_label(f'{PLOTVAR}{plot_units_label}')

def update(frame_number: int):
    frame_idx = int(frame_indices[frame_number])
    frame2d = _to_float(field.isel({time_dim: frame_idx}).transpose(y_dim, x_dim).values)

    ax.clear()
    mesh = ax.pcolormesh(
        x_plot,
        y_plot,
        np.ma.masked_invalid(frame2d),
        shading='auto',
        cmap=CMAP
        #,vmin=VMIN,
        #vmax=VMAX,
    )

    tval = np.asarray(times.isel({time_dim: frame_idx}).values)
    ax.set_title(f'{PLOTVAR}{plot_units_label}{layer_text} | {tval}')
    ax.set_xlabel('Longitude' if use_geo else x_dim)
    ax.set_ylabel('Latitude' if use_geo else y_dim)
    ax.grid(True, alpha=0.2)
    return (mesh,)

plt.rcParams['animation.embed_limit'] = 100  # MB, increase as needed
ani = FuncAnimation(fig, update, frames=len(frame_indices), interval=180, blit=False, repeat=False)
html_anim = HTML(ani.to_jshtml())
html_anim

In [ ]:
# ------------------------------ Cell 7: Chla animation for full year ------------------------------
# Goal: animate Chla at the top layer over the whole domain for all of 2015.
# Wet-cell mask: only show pixels where elevation > 0 at each timestep.

DATA_DIR = Path('/export/lv9/projects/dws/model_output/archived_runs/spinup_05/')
FILE_PATTERN = 'dws_500m.3d.2015??.nc'
PLOTVAR = 'Chla'
LAYER_INDEX = 10  # top=11, bottom=1
TIME_STEP = 1
CMAP = 'viridis'

files = sorted(DATA_DIR.glob(FILE_PATTERN))
if not files:
    raise FileNotFoundError(f'No files found with pattern: {FILE_PATTERN} in {DATA_DIR}')

ds = xr.open_mfdataset(
    files,
    combine='nested',
    concat_dim='time',
    decode_times=True,
    data_vars='minimal',
    coords='minimal',
    compat='override',
    join='override',
)

def _find_time_dim(da: xr.DataArray) -> str:
    for d in da.dims:
        if 'time' in d.lower():
            return d
    raise ValueError(f'No time-like dimension found in {da.dims}')

def _find_vertical_dim(da: xr.DataArray, time_dim: str) -> str | None:
    candidates = ('level', 'z', 'sigma', 'layer', 'lev', 'depth', 'nmesh2_layer_3d')
    for d in da.dims:
        if d == time_dim:
            continue
        if any(k in d.lower() for k in candidates):
            return d
    return None

def _pick_coord_name(ds: xr.Dataset, candidates: tuple[str, ...]) -> str | None:
    for name in candidates:
        if name in ds.variables:
            return name
    return None

def _to_float(values) -> np.ndarray:
    if np.ma.isMaskedArray(values):
        values = np.ma.filled(values, np.nan)
    return np.asarray(values, dtype=float)

def _drop_duplicate_time(da: xr.DataArray, time_dim: str) -> xr.DataArray:
    time_values = np.asarray(da[time_dim].values)
    _, keep_idx = np.unique(time_values, return_index=True)
    keep_idx = np.sort(keep_idx)
    if keep_idx.size < time_values.size:
        da = da.isel({time_dim: keep_idx})
    return da

if PLOTVAR not in ds.variables:
    raise KeyError(f"Variable '{PLOTVAR}' not found. Available: {sorted(ds.data_vars)}")

da = ds[PLOTVAR].squeeze(drop=True)
plot_units = da.attrs.get('units', '')
plot_units_label = f" [{plot_units}]" if plot_units else ''
time_dim = _find_time_dim(da)
z_dim = _find_vertical_dim(da, time_dim)

if z_dim is None:
    raise ValueError(f"No vertical dimension found in '{PLOTVAR}' dims {da.dims}")
if LAYER_INDEX >= da.sizes[z_dim] or LAYER_INDEX < -da.sizes[z_dim]:
    raise IndexError(f"LAYER_INDEX={LAYER_INDEX} out of bounds for '{z_dim}' size {da.sizes[z_dim]}")

field = da.isel({z_dim: LAYER_INDEX})
field = _drop_duplicate_time(field, time_dim)

# Derive colour range from the actual data
VMIN = 0
#float(field.min())
VMAX = 200  
#float(field.max())
print(f'Color range: {VMIN:.4f} - {VMAX:.4f}' + (' ' + plot_units if plot_units else ''))

xy_dims = [d for d in field.dims if d != time_dim]
if len(xy_dims) != 2:
    raise ValueError(f'Expected 2 horizontal dims, got {xy_dims}')
y_dim, x_dim = xy_dims

# Build elevation wet-cell mask: shape (time, y, x), True = wet
if 'elev' in ds.variables:
    elev_da = ds['elev'].squeeze(drop=True)
    elev_td = _find_time_dim(elev_da)
    elev_da = _drop_duplicate_time(elev_da, elev_td)
    # elev and field share the same time axis (same files); align by position
    n_frames = field.sizes[time_dim]
    elev_da = elev_da.isel({elev_td: slice(0, n_frames)})
    elev_mask = elev_da.transpose(elev_td, y_dim, x_dim).values > -1.1869  # (time, y, x) bool
    has_elev_mask = True
    print('Elevation mask loaded: only cells with elev > -1.1869 will be shown.')
else:
    has_elev_mask = False
    elev_mask = None
    print('Warning: elev not found – no wet-cell mask applied.')

times = field[time_dim]
frame_indices = np.arange(0, field.sizes[time_dim], TIME_STEP)
if len(frame_indices) == 0:
    raise ValueError('No frames selected. Check TIME_STEP and time length.')

ny = field.sizes[y_dim]
nx = field.sizes[x_dim]

lon_name = _pick_coord_name(ds, ('lonc', 'lon', 'longitude'))
lat_name = _pick_coord_name(ds, ('latc', 'lat', 'latitude'))

use_geo = False
if lon_name is not None and lat_name is not None:
    lon_da = ds[lon_name]
    lat_da = ds[lat_name]
    if y_dim in lon_da.dims and x_dim in lon_da.dims and y_dim in lat_da.dims and x_dim in lat_da.dims:
        x_geo = _to_float(lon_da.transpose(y_dim, x_dim).values)
        y_geo = _to_float(lat_da.transpose(y_dim, x_dim).values)
        if x_geo.shape == (ny, nx) and y_geo.shape == (ny, nx):
            if np.isfinite(x_geo).all() and np.isfinite(y_geo).all():
                x_plot, y_plot = x_geo, y_geo
                use_geo = True

if not use_geo:
    x_plot, y_plot = np.meshgrid(np.arange(nx, dtype=float), np.arange(ny, dtype=float))

fig, ax = plt.subplots(figsize=(8, 6))
plt.close(fig)

norm = plt.Normalize(vmin=VMIN, vmax=VMAX)
sm = plt.cm.ScalarMappable(norm=norm, cmap=CMAP)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, fraction=0.035, pad=0.03)
cbar.set_label(f'{PLOTVAR}{plot_units_label}')

def update(frame_number: int):
    frame_idx = int(frame_indices[frame_number])
    frame2d = _to_float(field.isel({time_dim: frame_idx}).transpose(y_dim, x_dim).values)

    # Apply wet-cell mask: dry cells → NaN
    if has_elev_mask:
        frame2d[~elev_mask[frame_idx]] = np.nan

    ax.clear()
    mesh = ax.pcolormesh(
        x_plot,
        y_plot,
        np.ma.masked_invalid(frame2d),
        shading='auto',
        cmap=CMAP,
        vmin=0,
        vmax=VMAX,
    )

    tval = np.asarray(times.isel({time_dim: frame_idx}).values)
    ax.set_title(f'{PLOTVAR}{plot_units_label} | layer {LAYER_INDEX} | {tval}')
    ax.set_xlabel('Longitude' if use_geo else x_dim)
    ax.set_ylabel('Latitude' if use_geo else y_dim)
    ax.grid(True, alpha=0.2)
    return (mesh,)

ani = FuncAnimation(fig, update, frames=len(frame_indices), interval=180, blit=False, repeat=False)
html_anim = HTML(ani.to_jshtml())
html_anim

In [ ]:
# ------------------------------ Cell 8: Chla animation for full year ------------------------------
# Goal: animate Chla at the top layer over the whole domain for all of 2015.

DATA_DIR = Path('/export/lv9/projects/dws/model_output/archived_runs/spinup_05/')
FILE_PATTERN = 'dws_500m.3d.2015??.nc'
PLOTVAR = 'Chla'
LAYER_INDEX = 10  # top=11, bottom=1
TIME_STEP = 1
VMIN = 0
VMAX = 200
CMAP = 'viridis'

files = sorted(DATA_DIR.glob(FILE_PATTERN))
if not files:
    raise FileNotFoundError(f'No files found with pattern: {FILE_PATTERN} in {DATA_DIR}')

# Concatenate in file order to avoid alignment errors from duplicate timestamps.
ds = xr.open_mfdataset(
    files,
    combine='nested',
    concat_dim='time',
    decode_times=True,
    data_vars='minimal',
    coords='minimal',
    compat='override',
    join='override',
)

def _find_time_dim(da: xr.DataArray) -> str:
    for d in da.dims:
        if 'time' in d.lower():
            return d
    raise ValueError(f'No time-like dimension found in {da.dims}')

def _find_vertical_dim(da: xr.DataArray, time_dim: str) -> str:
    candidates = ('level', 'z', 'sigma', 'layer', 'lev', 'depth', 'nmesh2_layer_3d')
    for d in da.dims:
        if d == time_dim:
            continue
        if any(k in d.lower() for k in candidates):
            return d
    if len(da.dims) >= 3:
        return da.dims[1]
    raise ValueError(f'No vertical-like dimension found in {da.dims}')

def _find_xy_dims(da: xr.DataArray, time_dim: str, z_dim: str) -> tuple[str, str]:
    remain = [d for d in da.dims if d not in (time_dim, z_dim)]
    if len(remain) != 2:
        raise ValueError(f'Expected 2 horizontal dims after removing time/z, got {remain} from {da.dims}')
    return remain[0], remain[1]

def _pick_coord_name(ds: xr.Dataset, candidates: tuple[str, ...]) -> str | None:
    for name in candidates:
        if name in ds.variables:
            return name
    return None

def _to_float(values) -> np.ndarray:
    if np.ma.isMaskedArray(values):
        values = np.ma.filled(values, np.nan)
    return np.asarray(values, dtype=float)

def _drop_duplicate_time(da: xr.DataArray, time_dim: str) -> xr.DataArray:
    time_values = np.asarray(da[time_dim].values)
    _, keep_idx = np.unique(time_values, return_index=True)
    keep_idx = np.sort(keep_idx)
    if keep_idx.size < time_values.size:
        da = da.isel({time_dim: keep_idx})
    return da

if PLOTVAR not in ds.variables:
    raise KeyError(f"Variable '{PLOTVAR}' not found. Available: {sorted(ds.data_vars)}")

da = ds[PLOTVAR]
plot_units = da.attrs.get('units', '')
plot_units_label = f" [{plot_units}]" if plot_units else ''
time_dim = _find_time_dim(da)
z_dim = _find_vertical_dim(da, time_dim)
y_dim, x_dim = _find_xy_dims(da, time_dim, z_dim)

if LAYER_INDEX >= da.sizes[z_dim] or LAYER_INDEX < -da.sizes[z_dim]:
    raise IndexError(f"LAYER_INDEX={LAYER_INDEX} out of bounds for '{z_dim}' size {da.sizes[z_dim]}")

field = da.isel({z_dim: LAYER_INDEX})
field = _drop_duplicate_time(field, time_dim)
times = field[time_dim]
frame_indices = np.arange(0, field.sizes[time_dim], TIME_STEP)
if len(frame_indices) == 0:
    raise ValueError('No frames selected. Check TIME_STEP and time length.')

ny = field.sizes[y_dim]
nx = field.sizes[x_dim]

lon_name = _pick_coord_name(ds, ('lonc', 'lon', 'longitude'))
lat_name = _pick_coord_name(ds, ('latc', 'lat', 'latitude'))

use_geo = False
if lon_name is not None and lat_name is not None:
    lon_da = ds[lon_name]
    lat_da = ds[lat_name]

    if y_dim in lon_da.dims and x_dim in lon_da.dims and y_dim in lat_da.dims and x_dim in lat_da.dims:
        x_geo = _to_float(lon_da.transpose(y_dim, x_dim).values)
        y_geo = _to_float(lat_da.transpose(y_dim, x_dim).values)

        if x_geo.shape == (ny, nx) and y_geo.shape == (ny, nx):
            if np.isfinite(x_geo).all() and np.isfinite(y_geo).all():
                x_plot, y_plot = x_geo, y_geo
                use_geo = True

if not use_geo:
    x_plot, y_plot = np.meshgrid(np.arange(nx, dtype=float), np.arange(ny, dtype=float))

fig, ax = plt.subplots(figsize=(8, 6))
plt.close(fig)

norm = plt.Normalize(vmin=VMIN, vmax=VMAX)
sm = plt.cm.ScalarMappable(norm=norm, cmap=CMAP)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, fraction=0.035, pad=0.03)
cbar.set_label(f'{PLOTVAR}{plot_units_label}')

def update(frame_number: int):
    frame_idx = int(frame_indices[frame_number])
    frame2d = _to_float(field.isel({time_dim: frame_idx}).transpose(y_dim, x_dim).values)

    ax.clear()
    mesh = ax.pcolormesh(
        x_plot,
        y_plot,
        np.ma.masked_invalid(frame2d),
        shading='auto',
        cmap=CMAP,
        vmin=VMIN,
        vmax=VMAX,
    )

    tval = np.asarray(times.isel({time_dim: frame_idx}).values)
    ax.set_title(f'{PLOTVAR}{plot_units_label} | layer {LAYER_INDEX} | {tval}')
    ax.set_xlabel('Longitude' if use_geo else x_dim)
    ax.set_ylabel('Latitude' if use_geo else y_dim)
    ax.grid(True, alpha=0.2)
    return (mesh,)

ani = FuncAnimation(fig, update, frames=len(frame_indices), interval=180, blit=False, repeat=False)
html_anim = HTML(ani.to_jshtml())
html_anim

In [ ]:
# ------------------------------ Cell 6 extra: Chla animation for full year ------------------------------
# Goal: animate Chla at the top layer over the whole domain for all of 2015.
# Add a mask for dry cells, by only plotting Chla where elevation data is available (elev)

DATA_DIR = Path('/export/lv9/projects/dws/model_output/archived_runs/spinup_05/')
FILE_PATTERN = 'dws_500m.3d.2015??.nc'
PLOTVAR = 'Chla'
LAYER_INDEX = 10  # top=11, bottom=1
TIME_STEP = 1
VMIN = 0
VMAX = 200
CMAP = 'viridis'

files = sorted(DATA_DIR.glob(FILE_PATTERN))
if not files:
    raise FileNotFoundError(f'No files found with pattern: {FILE_PATTERN} in {DATA_DIR}')

# Concatenate in file order to avoid alignment errors from duplicate timestamps.
ds = xr.open_mfdataset(
    files,
    combine='nested',
    concat_dim='time',
    decode_times=True,
    data_vars='minimal',
    coords='minimal',
    compat='override',
    join='override',
)

def _find_time_dim(da: xr.DataArray) -> str:
    for d in da.dims:
        if 'time' in d.lower():
            return d
    raise ValueError(f'No time-like dimension found in {da.dims}')

def _find_vertical_dim(da: xr.DataArray, time_dim: str) -> str:
    candidates = ('level', 'z', 'sigma', 'layer', 'lev', 'depth', 'nmesh2_layer_3d')
    for d in da.dims:
        if d == time_dim:
            continue
        if any(k in d.lower() for k in candidates):
            return d
    if len(da.dims) >= 3:
        return da.dims[1]
    raise ValueError(f'No vertical-like dimension found in {da.dims}')

def _find_xy_dims(da: xr.DataArray, time_dim: str, z_dim: str) -> tuple[str, str]:
    remain = [d for d in da.dims if d not in (time_dim, z_dim)]
    if len(remain) != 2:
        raise ValueError(f'Expected 2 horizontal dims after removing time/z, got {remain} from {da.dims}')
    return remain[0], remain[1]

def _pick_coord_name(ds: xr.Dataset, candidates: tuple[str, ...]) -> str | None:
    for name in candidates:
        if name in ds.variables:
            return name
    return None

def _to_float(values) -> np.ndarray:
    if np.ma.isMaskedArray(values):
        values = np.ma.filled(values, np.nan)
    return np.asarray(values, dtype=float)

def _drop_duplicate_time(da: xr.DataArray, time_dim: str) -> xr.DataArray:
    time_values = np.asarray(da[time_dim].values)
    _, keep_idx = np.unique(time_values, return_index=True)
    keep_idx = np.sort(keep_idx)
    if keep_idx.size < time_values.size:
        da = da.isel({time_dim: keep_idx})
    return da

if PLOTVAR not in ds.variables:
    raise KeyError(f"Variable '{PLOTVAR}' not found. Available: {sorted(ds.data_vars)}")

da = ds[PLOTVAR]
plot_units = da.attrs.get('units', '')
plot_units_label = f" [{plot_units}]" if plot_units else ''
time_dim = _find_time_dim(da)
z_dim = _find_vertical_dim(da, time_dim)
y_dim, x_dim = _find_xy_dims(da, time_dim, z_dim)

if LAYER_INDEX >= da.sizes[z_dim] or LAYER_INDEX < -da.sizes[z_dim]:
    raise IndexError(f"LAYER_INDEX={LAYER_INDEX} out of bounds for '{z_dim}' size {da.sizes[z_dim]}")

field = da.isel({z_dim: LAYER_INDEX})
field = _drop_duplicate_time(field, time_dim)
times = field[time_dim]
frame_indices = np.arange(0, field.sizes[time_dim], TIME_STEP)
if len(frame_indices) == 0:
    raise ValueError('No frames selected. Check TIME_STEP and time length.')

ny = field.sizes[y_dim]
nx = field.sizes[x_dim]

lon_name = _pick_coord_name(ds, ('lonc', 'lon', 'longitude'))
lat_name = _pick_coord_name(ds, ('latc', 'lat', 'latitude'))

use_geo = False
if lon_name is not None and lat_name is not None:
    lon_da = ds[lon_name]
    lat_da = ds[lat_name]

    if y_dim in lon_da.dims and x_dim in lon_da.dims and y_dim in lat_da.dims and x_dim in lat_da.dims:
        x_geo = _to_float(lon_da.transpose(y_dim, x_dim).values)
        y_geo = _to_float(lat_da.transpose(y_dim, x_dim).values)

        if x_geo.shape == (ny, nx) and y_geo.shape == (ny, nx):
            if np.isfinite(x_geo).all() and np.isfinite(y_geo).all():
                x_plot, y_plot = x_geo, y_geo
                use_geo = True

if not use_geo:
    x_plot, y_plot = np.meshgrid(np.arange(nx, dtype=float), np.arange(ny, dtype=float))

fig, ax = plt.subplots(figsize=(8, 6))
plt.close(fig)

norm = plt.Normalize(vmin=VMIN, vmax=VMAX)
sm = plt.cm.ScalarMappable(norm=norm, cmap=CMAP)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, fraction=0.035, pad=0.03)
cbar.set_label(f'{PLOTVAR}{plot_units_label}')

def update(frame_number: int):
    frame_idx = int(frame_indices[frame_number])
    frame2d = _to_float(field.isel({time_dim: frame_idx}).transpose(y_dim, x_dim).values)

    ax.clear()
    mesh = ax.pcolormesh(
        x_plot,
        y_plot,
        np.ma.masked_invalid(frame2d),
        shading='auto',
        cmap=CMAP,
        vmin=VMIN,
        vmax=VMAX,
    )

    tval = np.asarray(times.isel({time_dim: frame_idx}).values)
    ax.set_title(f'{PLOTVAR}{plot_units_label} | layer {LAYER_INDEX} | {tval}')
    ax.set_xlabel('Longitude' if use_geo else x_dim)
    ax.set_ylabel('Latitude' if use_geo else y_dim)
    ax.grid(True, alpha=0.2)
    return (mesh,)

ani = FuncAnimation(fig, update, frames=len(frame_indices), interval=180, blit=False, repeat=False)
html_anim = HTML(ani.to_jshtml())
html_anim

In [ ]:
# ------------------------------ Cell 7: Subdomain location on bathymetry ------------------------------
# Goal: show the selected subdomain on a bathymetry map.
DATA_DIR = Path('/export/lv9/projects/dws/model_output/archived_runs/spinup_05/')
FILE_PATTERN = 'dws_500m.3d.2015??.nc'

CHLA_VAR = 'Chla'
ELEV_VAR = 'elev'
Bathymetry_VAR = 'bathymetry'  # Preferred name for bathymetry variable; will try alternatives if not found.

# Subdomain index range (Python slice: start inclusive, stop exclusive).
# Salt marsh zone nearby Miedema (unusual discontinuity in derived total Chla)
#X_SLICE = (215, 225)
#Y_SLICE = (133, 136)

# Marsdiep zone
X_SLICE = (75, 78)
Y_SLICE = (95, 98)


# Reuse opened dataset if available; otherwise open yearly files.
try:
    ds
except NameError:
    files = sorted(DATA_DIR.glob(FILE_PATTERN))
    if not files:
        raise FileNotFoundError(f'No files found with pattern: {FILE_PATTERN} in {DATA_DIR}')
    ds = xr.open_mfdataset(
        files,
        combine='nested',
        concat_dim='time',
        decode_times=True,
        data_vars='minimal',
        coords='minimal',
        compat='override',
        join='override',
    )

def _find_bathy_name(ds: xr.Dataset, preferred: str) -> str:
    if preferred in ds.variables:
        return preferred
    candidates = ('bathymetry', 'depth', 'h', 'H', 'bathy', 'bat', 'topo', 'd', 'water_depth')
    for name in candidates:
        if name in ds.variables:
            return name
    raise KeyError(
        f"Bathymetry variable not found. Tried '{preferred}' and {candidates}. "
        f"Available vars include: {list(ds.variables)[:30]}"
    )

def _to_float(values) -> np.ndarray:
    if np.ma.isMaskedArray(values):
        values = np.ma.filled(values, np.nan)
    return np.asarray(values, dtype=float)

def _pick_coord_name(ds: xr.Dataset, candidates: tuple[str, ...]) -> str | None:
    for name in candidates:
        if name in ds.variables:
            return name
    return None

bathy_name = _find_bathy_name(ds, Bathymetry_VAR)
bathy = ds[bathy_name].squeeze(drop=True)

# Find horizontal dims from bathymetry; if a time dim exists, use first timestep.
bathy_dims = list(bathy.dims)
time_like = [d for d in bathy_dims if 'time' in d.lower()]
if time_like:
    bathy = bathy.isel({time_like[0]: 0})
    bathy_dims = [d for d in bathy.dims if d != time_like[0]]

if len(bathy_dims) != 2:
    raise ValueError(f'Expected 2D bathymetry after squeezing, got dims {bathy.dims}')
y_dim, x_dim = bathy_dims

ny = bathy.sizes[y_dim]
nx = bathy.sizes[x_dim]
y0, y1 = Y_SLICE
x0, x1 = X_SLICE
if not (0 <= y0 < y1 <= ny):
    raise IndexError(f'Y_SLICE={Y_SLICE} out of bounds for {y_dim} size {ny}')
if not (0 <= x0 < x1 <= nx):
    raise IndexError(f'X_SLICE={X_SLICE} out of bounds for {x_dim} size {nx}')

lon_name = _pick_coord_name(ds, ('lonc', 'lon', 'longitude'))
lat_name = _pick_coord_name(ds, ('latc', 'lat', 'latitude'))

use_geo = False
if lon_name is not None and lat_name is not None:
    lon_da = ds[lon_name]
    lat_da = ds[lat_name]
    if y_dim in lon_da.dims and x_dim in lon_da.dims and y_dim in lat_da.dims and x_dim in lat_da.dims:
        x_plot = _to_float(lon_da.transpose(y_dim, x_dim).values)
        y_plot = _to_float(lat_da.transpose(y_dim, x_dim).values)
        if x_plot.shape == (ny, nx) and y_plot.shape == (ny, nx):
            if np.isfinite(x_plot).all() and np.isfinite(y_plot).all():
                use_geo = True

if not use_geo:
    x_plot, y_plot = np.meshgrid(np.arange(nx, dtype=float), np.arange(ny, dtype=float))

bathy2d = _to_float(bathy.transpose(y_dim, x_dim).values)

fig, ax = plt.subplots(figsize=(8.6, 6.8))
mesh = ax.pcolormesh(
    x_plot,
    y_plot,
    np.ma.masked_invalid(bathy2d),
    shading='auto',
    cmap='cividis',
)
cbar = fig.colorbar(mesh, ax=ax, fraction=0.04, pad=0.03)
units = bathy.attrs.get('units', '')
cbar.set_label(f'{bathy_name} [{units}]' if units else bathy_name)

# Draw the selected subdomain as a closed polygon on the map.
if use_geo:
    x_poly = [x_plot[y0, x0], x_plot[y0, x1 - 1], x_plot[y1 - 1, x1 - 1], x_plot[y1 - 1, x0], x_plot[y0, x0]]
    y_poly = [y_plot[y0, x0], y_plot[y0, x1 - 1], y_plot[y1 - 1, x1 - 1], y_plot[y1 - 1, x0], y_plot[y0, x0]]
    ax.plot(x_poly, y_poly, color='red', lw=2.2, label='Selected subdomain')
else:
    x_poly = [x0, x1, x1, x0, x0]
    y_poly = [y0, y0, y1, y1, y0]
    ax.plot(x_poly, y_poly, color='red', lw=2.2, label='Selected subdomain')

ax.set_title('Subdomain location on bathymetry map')
ax.set_xlabel('Longitude' if use_geo else x_dim)
ax.set_ylabel('Latitude' if use_geo else y_dim)
ax.grid(True, alpha=0.2)
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

print(f'Bathymetry variable used: {bathy_name}')
print(f'Subdomain used: y={Y_SLICE[0]}:{Y_SLICE[1]}, x={X_SLICE[0]}:{X_SLICE[1]}')

In [ ]:
# ------------------------------ Cell 8: Subdomain dynamics (Chla vs elev) ------------------------------
# Goal: inspect temporal dynamics of surface-layer Chla, adjacent layers, and elevation for a selected area.

# Chla layer index to inspect and neighboring layers.
CHLA_LAYER_INDEX = 5  # top=11, bottom=1 in your convention
USE_DAILY_MEAN = False
ROLLING_WINDOW = None  # e.g., 3 for smoothing, or None

# Reuse opened dataset if available; otherwise open yearly files.
try:
    ds
except NameError:
    files = sorted(DATA_DIR.glob(FILE_PATTERN))
    if not files:
        raise FileNotFoundError(f'No files found with pattern: {FILE_PATTERN} in {DATA_DIR}')
    ds = xr.open_mfdataset(
        files,
        combine='nested',
        concat_dim='time',
        decode_times=True,
        data_vars='minimal',
        coords='minimal',
        compat='override',
        join='override',
    )

def _find_time_dim(da: xr.DataArray) -> str:
    for d in da.dims:
        if 'time' in d.lower():
            return d
    raise ValueError(f'No time dimension found in {da.dims}')

def _find_vertical_dim(da: xr.DataArray, time_dim: str) -> str | None:
    candidates = ('level', 'z', 'sigma', 'layer', 'lev', 'depth', 'nmesh2_layer_3d')
    for d in da.dims:
        if d != time_dim and any(k in d.lower() for k in candidates):
            return d
    return None

def _drop_duplicate_time(da: xr.DataArray, time_dim: str) -> xr.DataArray:
    time_values = np.asarray(da[time_dim].values)
    _, keep_idx = np.unique(time_values, return_index=True)
    keep_idx = np.sort(keep_idx)
    if keep_idx.size < time_values.size:
        da = da.isel({time_dim: keep_idx})
    return da

def _maybe_smooth(series: xr.DataArray, time_dim: str) -> xr.DataArray:
    out = series
    if USE_DAILY_MEAN:
        out = out.resample({time_dim: '1D'}).mean(skipna=True)
    if ROLLING_WINDOW is not None:
        if ROLLING_WINDOW < 1:
            raise ValueError('ROLLING_WINDOW must be >= 1 or None')
        out = out.rolling({time_dim: ROLLING_WINDOW}, center=True).mean()
    return out

if CHLA_VAR not in ds.variables:
    raise KeyError(f"Variable '{CHLA_VAR}' not found. Available: {sorted(ds.data_vars)}")
if ELEV_VAR not in ds.variables:
    raise KeyError(f"Variable '{ELEV_VAR}' not found. Available: {sorted(ds.data_vars)}")

chla = ds[CHLA_VAR].squeeze(drop=True)
elev = ds[ELEV_VAR].squeeze(drop=True)

time_dim = _find_time_dim(chla)
z_dim = _find_vertical_dim(chla, time_dim)
if z_dim is None:
    raise ValueError(f"No vertical dimension found for {CHLA_VAR}. Dims: {chla.dims}")

xy_dims = [d for d in chla.dims if d not in (time_dim, z_dim)]
if len(xy_dims) != 2:
    raise ValueError(f'Expected 2 horizontal dims for {CHLA_VAR}, got {xy_dims} from {chla.dims}')
y_dim, x_dim = xy_dims

y0, y1 = Y_SLICE
x0, x1 = X_SLICE
if not (0 <= y0 < y1 <= chla.sizes[y_dim]):
    raise IndexError(f'Y_SLICE={Y_SLICE} out of bounds for {y_dim} size {chla.sizes[y_dim]}')
if not (0 <= x0 < x1 <= chla.sizes[x_dim]):
    raise IndexError(f'X_SLICE={X_SLICE} out of bounds for {x_dim} size {chla.sizes[x_dim]}')

if CHLA_LAYER_INDEX >= chla.sizes[z_dim] or CHLA_LAYER_INDEX < -chla.sizes[z_dim]:
    raise IndexError(f"CHLA_LAYER_INDEX={CHLA_LAYER_INDEX} out of bounds for '{z_dim}' size {chla.sizes[z_dim]}")

layer_center = 5
layer_up = 10
layer_down = 1

chla_sub = chla.isel({y_dim: slice(y0, y1), x_dim: slice(x0, x1)})
elev_sub = elev.isel({y_dim: slice(y0, y1), x_dim: slice(x0, x1)})

chla_surface_series = chla_sub.isel({z_dim: layer_center}).mean(dim=(y_dim, x_dim), skipna=True)
chla_up_series = chla_sub.isel({z_dim: layer_up}).mean(dim=(y_dim, x_dim), skipna=True)
chla_down_series = chla_sub.isel({z_dim: layer_down}).mean(dim=(y_dim, x_dim), skipna=True)

elev_time_dim = _find_time_dim(elev_sub)
elev_spatial_dims = [d for d in elev_sub.dims if d != elev_time_dim]
if len(elev_spatial_dims) != 2:
    raise ValueError(f'Expected 2 horizontal dims for {ELEV_VAR}, got {elev_spatial_dims} from {elev_sub.dims}')
elev_series = elev_sub.mean(dim=elev_spatial_dims, skipna=True)

chla_surface_series = _drop_duplicate_time(chla_surface_series, time_dim)
chla_up_series = _drop_duplicate_time(chla_up_series, time_dim)
chla_down_series = _drop_duplicate_time(chla_down_series, time_dim)
elev_series = _drop_duplicate_time(elev_series, elev_time_dim)

chla_surface_series = _maybe_smooth(chla_surface_series, time_dim)
chla_up_series = _maybe_smooth(chla_up_series, time_dim)
chla_down_series = _maybe_smooth(chla_down_series, time_dim)
elev_series = _maybe_smooth(elev_series, elev_time_dim)

# Treat physically invalid negative Chla values as missing.
chla_surface_series = chla_surface_series.where(chla_surface_series >= 0)
chla_up_series = chla_up_series.where(chla_up_series >= 0)
chla_down_series = chla_down_series.where(chla_down_series >= 0)

# Plot 1: Chla at selected/adjacent layers + elevation on twin axis.
fig, ax1 = plt.subplots(figsize=(12, 4.8))
ax2 = ax1.twinx()

ax1.plot(chla_surface_series[time_dim].values, chla_surface_series.values, lw=2.0, label=f'{CHLA_VAR} layer {layer_center}')
ax1.plot(chla_up_series[time_dim].values, chla_up_series.values, lw=1.4, ls='--', label=f'{CHLA_VAR} layer up ({layer_up})')
ax1.plot(chla_down_series[time_dim].values, chla_down_series.values, lw=1.4, ls=':', label=f'{CHLA_VAR} layer down ({layer_down})')
ax2.plot(elev_series[elev_time_dim].values, elev_series.values, color='black', lw=1.3, alpha=0.65, label=ELEV_VAR)

chla_units = chla.attrs.get('units', '')
elev_units = elev.attrs.get('units', '')
ax1.set_ylabel(f'{CHLA_VAR} [{chla_units}]' if chla_units else CHLA_VAR)
ax2.set_ylabel(f'{ELEV_VAR} [{elev_units}]' if elev_units else ELEV_VAR)
ax1.set_xlabel('Time')
ax1.grid(True, alpha=0.25)
ax1.set_title(
    f'Subdomain dynamics | y={Y_SLICE[0]}:{Y_SLICE[1]}, x={X_SLICE[0]}:{X_SLICE[1]} | '
    f'{CHLA_VAR} (surface/up/down) and {ELEV_VAR}'
)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', ncol=2, fontsize=9)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print(f'Subdomain used: y={Y_SLICE[0]}:{Y_SLICE[1]}, x={X_SLICE[0]}:{X_SLICE[1]}')


In [ ]:
# ------------------------------ Cell 8: 
# Goal: Compare model results with observations for temporal dynamics of Chla and other variables nearby Marsdiep.

import pandas as pd
import matplotlib.pyplot as plt

# The time series data for Marsdiep is available from the NIOZ Dataverse at http://doi.org/10.25850/nioz/7b.b.5j

Validation_DATA_DIR = Path('/export/lv9/projects/dws/results/validation/pelagic/')
Marsdiep_ts = 'Jetty_HWseries.csv'
csv_path = Validation_DATA_DIR / Marsdiep_ts

if not csv_path.exists():
    raise FileNotFoundError(f'CSV file not found: {csv_path}')

measurement_df = pd.read_csv(
    csv_path,
    na_values=['NA', ''],
    parse_dates=['timestamp'],
)

print(f'Loaded CSV: {csv_path}')
print(f'Shape: {measurement_df.shape[0]} rows x {measurement_df.shape[1]} columns')
print('Columns:')
print(list(measurement_df.columns))
print('')
print('First 5 rows:')
display(measurement_df.head())


# Parameters for model-observation comparison

MODEL_SURFACE_LAYER_INDEX = 10  # top=11, bottom=1 in your convention
USE_DAILY_MEAN = False
ROLLING_WINDOW = None  # e.g. 3 for smoothing, or None

# Reuse opened dataset if available; otherwise open yearly files.
try:
    ds
except NameError:
    files = sorted(DATA_DIR.glob(FILE_PATTERN))
    if not files:
        raise FileNotFoundError(f'No files found with pattern: {FILE_PATTERN} in {DATA_DIR}')
    ds = xr.open_mfdataset(
        files,
        combine='nested',
        concat_dim='time',
        decode_times=True,
        data_vars='minimal',
        coords='minimal',
        compat='override',
        join='override',
    )

def _find_time_dim(da: xr.DataArray) -> str:
    for d in da.dims:
        if 'time' in d.lower():
            return d
    raise ValueError(f'No time dimension found in {da.dims}')

def _find_vertical_dim(da: xr.DataArray, time_dim: str) -> str | None:
    candidates = ('level', 'z', 'sigma', 'layer', 'lev', 'depth', 'nmesh2_layer_3d')
    for d in da.dims:
        if d != time_dim and any(k in d.lower() for k in candidates):
            return d
    return None

def _drop_duplicate_time(da: xr.DataArray, time_dim: str) -> xr.DataArray:
    time_values = np.asarray(da[time_dim].values)
    _, keep_idx = np.unique(time_values, return_index=True)
    keep_idx = np.sort(keep_idx)
    if keep_idx.size < time_values.size:
        da = da.isel({time_dim: keep_idx})
    return da

def _maybe_smooth(series: xr.DataArray, time_dim: str) -> xr.DataArray:
    out = series
    if USE_DAILY_MEAN:
        out = out.resample({time_dim: '1D'}).mean(skipna=True)
    if ROLLING_WINDOW is not None:
        if ROLLING_WINDOW < 1:
            raise ValueError('ROLLING_WINDOW must be >= 1 or None')
        out = out.rolling({time_dim: ROLLING_WINDOW}, center=True).mean()
    return out

if CHLA_VAR not in ds.variables:
    raise KeyError(f"Variable '{CHLA_VAR}' not found. Available: {sorted(ds.data_vars)}")

chla = ds[CHLA_VAR].squeeze(drop=True)
time_dim = _find_time_dim(chla)
z_dim = _find_vertical_dim(chla, time_dim)
if z_dim is None:
    raise ValueError(f"No vertical dimension found for {CHLA_VAR}. Dims: {chla.dims}")

xy_dims = [d for d in chla.dims if d not in (time_dim, z_dim)]
if len(xy_dims) != 2:
    raise ValueError(f'Expected 2 horizontal dims for {CHLA_VAR}, got {xy_dims} from {chla.dims}')
y_dim, x_dim = xy_dims

y0, y1 = Y_SLICE
x0, x1 = X_SLICE
if not (0 <= y0 < y1 <= chla.sizes[y_dim]):
    raise IndexError(f'Y_SLICE={Y_SLICE} out of bounds for {y_dim} size {chla.sizes[y_dim]}')
if not (0 <= x0 < x1 <= chla.sizes[x_dim]):
    raise IndexError(f'X_SLICE={X_SLICE} out of bounds for {x_dim} size {chla.sizes[x_dim]}')

if MODEL_SURFACE_LAYER_INDEX >= chla.sizes[z_dim] or MODEL_SURFACE_LAYER_INDEX < -chla.sizes[z_dim]:
    raise IndexError(
        f"MODEL_SURFACE_LAYER_INDEX={MODEL_SURFACE_LAYER_INDEX} out of bounds for '{z_dim}' size {chla.sizes[z_dim]}"
    )

# Model: spatial mean over the selected subdomain at the surface layer.
chla_sub = chla.isel({y_dim: slice(y0, y1), x_dim: slice(x0, x1)})
model_series = chla_sub.isel({z_dim: MODEL_SURFACE_LAYER_INDEX}).mean(dim=(y_dim, x_dim), skipna=True)
model_series = _drop_duplicate_time(model_series, time_dim)
model_series = _maybe_smooth(model_series, time_dim)
model_series = model_series.where(model_series >= 0)

# Observations: reuse measurement_df from Cell 9 if present, otherwise load it.
try:
    measurement_df
except NameError:
    Validation_DATA_DIR = Path('/export/lv9/projects/dws/results/validation/pelagic/')
    Marsdiep_ts = 'Jetty_HWseries.csv'
    csv_path = Validation_DATA_DIR / Marsdiep_ts
    if not csv_path.exists():
        raise FileNotFoundError(f'CSV file not found: {csv_path}')
    measurement_df = pd.read_csv(
        csv_path,
        na_values=['NA', ''],
        parse_dates=['timestamp'],
    )

if 'Chl' not in measurement_df.columns:
    raise KeyError("Column 'Chl' not found in the CSV file.")

obs_df = measurement_df[['timestamp', 'Chl']].dropna(subset=['timestamp', 'Chl']).copy()
obs_df = obs_df.sort_values('timestamp')
obs_df = obs_df[obs_df['timestamp'].dt.year == 2015]

if obs_df.empty:
    raise ValueError('No Chl measurements found for 2015.')

# Plot model and observations together.
fig, ax = plt.subplots(figsize=(12, 4.8))

ax.plot(
    model_series[time_dim].values,
    model_series.values,
    lw=2.0,
    color='tab:blue',
    label=f'Model {CHLA_VAR} surface layer {MODEL_SURFACE_LAYER_INDEX}',
)

ax.scatter(
    obs_df['timestamp'],
    obs_df['Chl'],
    s=22,
    color='tab:orange',
    alpha=0.85,
    label='Observed surface Chl',
    zorder=3,
)

chla_units = chla.attrs.get('units', '')
ax.set_ylabel(f'{CHLA_VAR} / Chl [{chla_units}]' if chla_units else f'{CHLA_VAR} / Chl')
ax.set_xlabel('Time')
ax.set_title(
    f'Model vs observed surface chlorophyll in 2015 | '
    f'y={Y_SLICE[0]}:{Y_SLICE[1]}, x={X_SLICE[0]}:{X_SLICE[1]}'
)
ax.grid(True, alpha=0.25)
ax.legend(loc='upper left')
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print(f'Subdomain used: y={Y_SLICE[0]}:{Y_SLICE[1]}, x={X_SLICE[0]}:{X_SLICE[1]}')
print(f'Model surface layer used: {MODEL_SURFACE_LAYER_INDEX}')
print(f'Number of observation points in 2015: {len(obs_df)}')

In [ ]:
# ------------------------------ Cell 9: Model vs observations (surface, 2015) ------------------------------
# Compare model surface-layer subdomain means with observed surface measurements for:
# Chla (Chl), Phosphate (N1p vs PO4), Nitrate (N3n vs NO3), Ammonium (N4n vs NH4)

import pandas as pd
import matplotlib.pyplot as plt

MODEL_SURFACE_LAYER_INDEX = 10  # top=11, bottom=1 in your convention
USE_DAILY_MEAN = False
ROLLING_WINDOW = None  # e.g. 3 for smoothing, or None

# Model variable -> observation column
VAR_MAP = {
    "Chla": "Chl",
    "N1p": "PO4",
    "N3n": "NO3",
    "N4n": "NH4",
}

# Reuse opened dataset if available; otherwise open yearly files.
try:
    ds
except NameError:
    files = sorted(DATA_DIR.glob(FILE_PATTERN))
    if not files:
        raise FileNotFoundError(f"No files found with pattern: {FILE_PATTERN} in {DATA_DIR}")
    ds = xr.open_mfdataset(
        files,
        combine="nested",
        concat_dim="time",
        decode_times=True,
        data_vars="minimal",
        coords="minimal",
        compat="override",
        join="override",
    )

def _find_time_dim(da: xr.DataArray) -> str:
    for d in da.dims:
        if "time" in d.lower():
            return d
    raise ValueError(f"No time dimension found in {da.dims}")

def _find_vertical_dim(da: xr.DataArray, time_dim: str) -> str | None:
    candidates = ("level", "z", "sigma", "layer", "lev", "depth", "nmesh2_layer_3d")
    for d in da.dims:
        if d != time_dim and any(k in d.lower() for k in candidates):
            return d
    return None

def _drop_duplicate_time(da: xr.DataArray, time_dim: str) -> xr.DataArray:
    time_values = np.asarray(da[time_dim].values)
    _, keep_idx = np.unique(time_values, return_index=True)
    keep_idx = np.sort(keep_idx)
    if keep_idx.size < time_values.size:
        da = da.isel({time_dim: keep_idx})
    return da

def _maybe_smooth(series: xr.DataArray, time_dim: str) -> xr.DataArray:
    out = series
    if USE_DAILY_MEAN:
        out = out.resample({time_dim: "1D"}).mean(skipna=True)
    if ROLLING_WINDOW is not None:
        if ROLLING_WINDOW < 1:
            raise ValueError("ROLLING_WINDOW must be >= 1 or None")
        out = out.rolling({time_dim: ROLLING_WINDOW}, center=True).mean()
    return out

def _model_surface_series(ds: xr.Dataset, model_var: str, layer_index: int) -> tuple[xr.DataArray, str]:
    if model_var not in ds.variables:
        raise KeyError(f"Model variable '{model_var}' not found. Available: {sorted(ds.data_vars)}")

    da = ds[model_var].squeeze(drop=True)
    time_dim = _find_time_dim(da)
    z_dim = _find_vertical_dim(da, time_dim)
    if z_dim is None:
        raise ValueError(f"No vertical dimension found for {model_var}. Dims: {da.dims}")

    xy_dims = [d for d in da.dims if d not in (time_dim, z_dim)]
    if len(xy_dims) != 2:
        raise ValueError(f"Expected 2 horizontal dims for {model_var}, got {xy_dims} from {da.dims}")
    y_dim, x_dim = xy_dims

    y0, y1 = Y_SLICE
    x0, x1 = X_SLICE
    if not (0 <= y0 < y1 <= da.sizes[y_dim]):
        raise IndexError(f"Y_SLICE={Y_SLICE} out of bounds for {y_dim} size {da.sizes[y_dim]}")
    if not (0 <= x0 < x1 <= da.sizes[x_dim]):
        raise IndexError(f"X_SLICE={X_SLICE} out of bounds for {x_dim} size {da.sizes[x_dim]}")

    if layer_index >= da.sizes[z_dim] or layer_index < -da.sizes[z_dim]:
        raise IndexError(f"layer_index={layer_index} out of bounds for '{z_dim}' size {da.sizes[z_dim]}")

    sub = da.isel({y_dim: slice(y0, y1), x_dim: slice(x0, x1)})
    series = sub.isel({z_dim: layer_index}).mean(dim=(y_dim, x_dim), skipna=True)
    series = _drop_duplicate_time(series, time_dim)
    series = _maybe_smooth(series, time_dim)
    return series, time_dim

# Observations: reuse measurement_df if present, otherwise load it.
try:
    measurement_df
except NameError:
    Validation_DATA_DIR = Path("/export/lv9/projects/dws/results/validation/pelagic/")
    Marsdiep_ts = "Jetty_HWseries.csv"
    csv_path = Validation_DATA_DIR / Marsdiep_ts
    if not csv_path.exists():
        raise FileNotFoundError(f"CSV file not found: {csv_path}")
    measurement_df = pd.read_csv(
        csv_path,
        na_values=["NA", ""],
        parse_dates=["timestamp"],
    )

if "timestamp" not in measurement_df.columns:
    raise KeyError("Column 'timestamp' not found in observations CSV.")

obs_df = measurement_df.copy()
obs_df = obs_df.dropna(subset=["timestamp"])
obs_df = obs_df[obs_df["timestamp"].dt.year == 2015].sort_values("timestamp")

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
axes = axes.ravel()

for i, (model_var, obs_col) in enumerate(VAR_MAP.items()):
    ax = axes[i]

    if obs_col not in obs_df.columns:
        ax.text(0.5, 0.5, f"Obs column '{obs_col}' not found", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(f"{model_var} vs {obs_col}")
        ax.set_axis_off()
        continue

    model_series, time_dim = _model_surface_series(ds, model_var, MODEL_SURFACE_LAYER_INDEX)
    model_series = model_series.where(model_series >= 0 if model_var == "Chla" else True)

    obs_var = obs_df[["timestamp", obs_col]].dropna()

    ax.plot(
        model_series[time_dim].values,
        model_series.values,
        lw=2.0,
        color="tab:blue",
        label=f"Model {model_var} (layer {MODEL_SURFACE_LAYER_INDEX})",
    )
    ax.scatter(
        obs_var["timestamp"],
        obs_var[obs_col],
        s=18,
        color="tab:orange",
        alpha=0.85,
        label=f"Obs {obs_col}",
        zorder=3,
    )

    units = ds[model_var].attrs.get("units", "")
    ax.set_ylabel(f"{model_var} / {obs_col} [{units}]" if units else f"{model_var} / {obs_col}")
    ax.set_title(f"{model_var} (model) vs {obs_col} (obs)")
    ax.grid(True, alpha=0.25)
    ax.legend(loc="upper left", fontsize=8)

for ax in axes:
    ax.set_xlabel("Time")

fig.suptitle(
    f"Model vs observed surface concentrations in 2015 | y={Y_SLICE[0]}:{Y_SLICE[1]}, x={X_SLICE[0]}:{X_SLICE[1]}",
    y=1.02,
)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------ Cell 10: Model vs observations (surface, 2015) ------------------------------
# Compare model subdomain means with observed measurements for:
# Primary production

import pandas as pd
import matplotlib.pyplot as plt

Marsdiep_PP_ts = 'PP_HW40_2012_2023.csv'
PP_csv_path = Validation_DATA_DIR / Marsdiep_PP_ts

if not PP_csv_path.exists():
    raise FileNotFoundError(f'CSV file not found: {PP_csv_path}')

pp_df = pd.read_csv(PP_csv_path, na_values=['NA', ''])
if 'Date' not in pp_df.columns or 'Daily_PP' not in pp_df.columns:
    raise KeyError("CSV must contain columns 'Date' and 'Daily_PP'.")

pp_df['Date'] = pd.to_datetime(pp_df['Date'], errors='coerce')
pp_obs = pp_df[['Date', 'Daily_PP']].dropna(subset=['Date', 'Daily_PP']).copy()
pp_obs = pp_obs.sort_values('Date')
pp_obs = pp_obs[pp_obs['Date'].dt.year == 2015]

if pp_obs.empty:
    raise ValueError('No Daily_PP observations found for 2015 in PP CSV.')

# Reuse opened dataset if available; otherwise open yearly files.
try:
    ds
except NameError:
    files = sorted(DATA_DIR.glob(FILE_PATTERN))
    if not files:
        raise FileNotFoundError(f'No files found with pattern: {FILE_PATTERN} in {DATA_DIR}')
    ds = xr.open_mfdataset(
        files,
        combine='nested',
        concat_dim='time',
        decode_times=True,
        data_vars='minimal',
        coords='minimal',
        compat='override',
        join='override',
    )

MODEL_PP_VAR = 'netPPm2'
if MODEL_PP_VAR not in ds.variables:
    raise KeyError(f"Variable '{MODEL_PP_VAR}' not found. Available: {sorted(ds.data_vars)}")

pp_model_da = ds[MODEL_PP_VAR].squeeze(drop=True)

def _find_time_dim(da: xr.DataArray) -> str:
    for d in da.dims:
        if 'time' in d.lower():
            return d
    raise ValueError(f'No time dimension found in {da.dims}')

def _drop_duplicate_time(da: xr.DataArray, time_dim: str) -> xr.DataArray:
    time_values = np.asarray(da[time_dim].values)
    _, keep_idx = np.unique(time_values, return_index=True)
    keep_idx = np.sort(keep_idx)
    if keep_idx.size < time_values.size:
        da = da.isel({time_dim: keep_idx})
    return da

time_dim = _find_time_dim(pp_model_da)
spatial_dims = [d for d in pp_model_da.dims if d != time_dim]

if len(spatial_dims) >= 2:
    # Use same subdomain as previous cells when 2D horizontal dims exist.
    y_dim, x_dim = spatial_dims[-2], spatial_dims[-1]
    y0, y1 = Y_SLICE
    x0, x1 = X_SLICE
    if 0 <= y0 < y1 <= pp_model_da.sizes[y_dim] and 0 <= x0 < x1 <= pp_model_da.sizes[x_dim]:
        pp_model_da = pp_model_da.isel({y_dim: slice(y0, y1), x_dim: slice(x0, x1)})

spatial_dims = [d for d in pp_model_da.dims if d != time_dim]
if spatial_dims:
    pp_model_series = pp_model_da.mean(dim=spatial_dims, skipna=True)
else:
    pp_model_series = pp_model_da

pp_model_series = _drop_duplicate_time(pp_model_series, time_dim)
# Remove physically invalid negative model PP values
pp_model_series = pp_model_series.where(pp_model_series >= 0)

# Match observation cadence: daily mean from model.
pp_model_daily = pp_model_series.resample({time_dim: '1D'}).mean(skipna=True)
pp_model_df = pd.DataFrame({
    'Date': pd.to_datetime(pp_model_daily[time_dim].values),
    'Model_PP': pp_model_daily.values
})
pp_model_df = pp_model_df.dropna(subset=['Date', 'Model_PP'])
pp_model_df = pp_model_df[pp_model_df['Date'].dt.year == 2015]

if pp_model_df.empty:
    raise ValueError('No model netPPm2 values found for 2015.')

# Plot model and observations.
fig, ax = plt.subplots(figsize=(12, 4.8))

ax.plot(
    pp_model_df['Date'],
    pp_model_df['Model_PP'],
    lw=2.0,
    color='tab:blue',
    label='Model netPPm2 (daily mean)',
)
ax.scatter(
    pp_obs['Date'],
    pp_obs['Daily_PP'],
    s=24,
    color='tab:orange',
    alpha=0.9,
    label='Observed Daily_PP',
    zorder=3,
)

pp_units = ds[MODEL_PP_VAR].attrs.get('units', '')
ax.set_ylabel(f'PP [{pp_units}]' if pp_units else 'PP')
ax.set_xlabel('Time')
ax.set_title(
    f'Model vs observed primary production in 2015 | y={Y_SLICE[0]}:{Y_SLICE[1]}, x={X_SLICE[0]}:{X_SLICE[1]}'
)
ax.grid(True, alpha=0.25)
ax.legend(loc='upper left')
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print(f'PP CSV used: {PP_csv_path}')
print(f'Observation points in 2015: {len(pp_obs)}')
print(f'Model daily points in 2015: {len(pp_model_df)}')

In [ ]:
# ── Side-by-side comparison: Satellite CHL vs Model Chla (surface layer) ──────
# Satellite: Copernicus CMEMS L4 gap-free daily CHL (regular lon/lat grid)
# Model:     GETM/ERSEM Chla, top layer, curvilinear grid
# Both are projected to Web Mercator (EPSG:3857) and overlaid on the same basemap.

SAT_FILE = "/export/lv9/projects/dws/results/validation/pelagic/satellite/copernicus_satellite_chl_2015.nc"
OUTPUT_GIF = Path("/export/lv9/projects/dws/results/validation/pelagic/satellite/model_vs_satellite_chl_2015.gif")
NETCDF_PATH = Path('/export/lv9/projects/dws/model_output/archived_runs/spinup_05/')

MODEL_CHLA_VAR = "Chla"
SAT_CHLA_VAR   = "CHL"          # variable name inside the satellite NetCDF
SURFACE_LAYER_CHLA = 10         # top layer index in the model (same as SURFACE_LAYER_INDEX)
TIME_STEP_COMP = 1              # step through satellite time axis (1 = every day)

CHLA_VMIN = 0.0
CHLA_VMAX = 30.0                # mg/m³  – adjust to your typical range
CHLA_CMAP = "viridis"

OUTPUT_COMPARISON_GIF = OUTPUT_GIF.with_name(f"{OUTPUT_GIF.stem}_chla_comparison.gif")


# ── 1. Open datasets ──────────────────────────────────────────────────────────
# Basemap settings (requires internet access and `contextily`)
# Install with: pip install contextily
USE_BASEMAP = False
BASEMAP_SOURCE = "https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}" # Google Satellite
BASEMAP_ATTRIBUTION = "Map data © Google"
BASEMAP_ZOOM = 8

# Fixed map window: Dutch Wadden Sea to Ems-Dollard (lon_min, lon_max, lat_min, lat_max)
MAP_EXTENT_LONLAT = (4.27, 7.6, 52.47, 53.6)

SURFACE_ALPHA = 0.72

if USE_BASEMAP:
    try:
        import contextily as ctx
        print(f"contextily version: {ctx.__version__}")
    except ImportError as exc:
        raise ImportError(
            "Basemap requested but contextily is not installed. "
            "Install with: pip install contextily"
        ) from exc


ds_sat   = xr.open_dataset(SAT_FILE, decode_times=True)

def _find_time_dim(da: xr.DataArray) -> str:
    for d in da.dims:
        if 'time' in d.lower():
            return d
    raise ValueError(f'No time dimension found in {da.dims}')

def _find_vertical_dim(da: xr.DataArray, time_dim: str) -> str | None:
    candidates = ('level', 'z', 'sigma', 'layer', 'lev', 'depth', 'nmesh2_layer_3d')
    for d in da.dims:
        if d != time_dim and any(k in d.lower() for k in candidates):
            return d
    return None

def _drop_duplicate_time(da: xr.DataArray, time_dim: str) -> xr.DataArray:
    time_values = np.asarray(da[time_dim].values)
    _, keep_idx = np.unique(time_values, return_index=True)
    keep_idx = np.sort(keep_idx)
    if keep_idx.size < time_values.size:
        da = da.isel({time_dim: keep_idx})
    return da

def _lonlat_to_webmercator(lon: np.ndarray, lat: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    r = 6378137.0
    lon = np.asarray(lon, dtype=float)
    lat = np.asarray(lat, dtype=float)
    lat = np.clip(lat, -85.05112878, 85.05112878)
    x = r * np.deg2rad(lon)
    y = r * np.log(np.tan(np.pi / 4.0 + np.deg2rad(lat) / 2.0))
    return x, y

def _find_xy_dims(da: xr.DataArray, time_dim: str, z_dim: str) -> tuple[str, str]:
    remain = [d for d in da.dims if d not in (time_dim, z_dim)]
    if len(remain) != 2:
        raise ValueError(
            f"Expected 2 horizontal dims after removing time/z, got {remain} from {da.dims}"
        )
    return remain[0], remain[1]

def _pick_coord_name(ds: xr.Dataset, candidates: tuple[str, ...]) -> str | None:
    for name in candidates:
        if name in ds.variables:
            return name
    return None

def _safe_fill_coords(arr: np.ndarray) -> np.ndarray:
    """Fill NaN/Inf in a 2D coordinate array by forward/backward fill along rows and columns.

    Land cells in GETM curvilinear grids often carry NaN in lon/lat.  pcolormesh cannot
    accept non-finite coordinate values, but those cells will be masked in the data array
    anyway, so any reasonable fill value is fine.
    """
    import pandas as pd
    arr = np.array(arr, dtype=float)
    arr[~np.isfinite(arr)] = np.nan
    df = pd.DataFrame(arr)
    filled = df.ffill(axis=1).bfill(axis=1).ffill(axis=0).bfill(axis=0)
    return np.asarray(filled.values, dtype=float)


topo_file = Path('/export/lv9/projects/dws/model_input/bathymetry/topo_dws_500m.nc')
topo_ds = xr.open_dataset(topo_file)
xc = topo_ds['xc'].values
yc = topo_ds['yc'].values

DATA_DIR = Path('/export/lv9/projects/dws/model_output/archived_runs/spinup_05/')
FILE_PATTERN = 'dws_500m.3d.2015??.nc'

# Crop bounds from topo file
XC_MIN, XC_MAX = float(xc.min()), float(xc.max())
YC_MIN, YC_MAX = float(yc.min()), float(yc.max())
print(f'Topo-derived crop bounds — xc: {XC_MIN:.0f}–{XC_MAX:.0f}, yc: {YC_MIN:.0f}–{YC_MAX:.0f}')

files = sorted(DATA_DIR.glob(FILE_PATTERN))
if not files:
    raise FileNotFoundError(f'No files found: {FILE_PATTERN} in {DATA_DIR}')
    
FILE_PATTERN = 'dws_500m.3d.2015??.nc'   # adjust pattern to match your files
model_files = sorted(NETCDF_PATH.glob(FILE_PATTERN))
if not model_files:
    raise FileNotFoundError(f"No files found matching {FILE_PATTERN} in {NETCDF_PATH}")
ds_model = xr.open_mfdataset(
    model_files,
    combine="nested",
    concat_dim="time",
    decode_times=True,
    data_vars="minimal",
    coords="minimal",
    compat="override",
    join="override",
)

# ── 2. Extract satellite CHL ──────────────────────────────────────────────────
sat_da = ds_sat[SAT_CHLA_VAR]
sat_time_dim = _find_time_dim(sat_da)
sat_times = sat_da[sat_time_dim]

# Satellite lon/lat are 1-D (regular grid) → build 2-D meshgrid
sat_lon_name = next((n for n in ("longitude", "lon") if n in ds_sat.variables), None)
sat_lat_name = next((n for n in ("latitude",  "lat") if n in ds_sat.variables), None)
if sat_lon_name is None or sat_lat_name is None:
    raise KeyError(f"Cannot find lon/lat in satellite file. Variables: {list(ds_sat.variables)}")

sat_lon1d = np.asarray(ds_sat[sat_lon_name].values, dtype=float)
sat_lat1d = np.asarray(ds_sat[sat_lat_name].values, dtype=float)
sat_lon2d, sat_lat2d = np.meshgrid(sat_lon1d, sat_lat1d)   # (lat, lon)
sat_x_map, sat_y_map = _lonlat_to_webmercator(sat_lon2d, sat_lat2d)

# ── 3. Extract model Chla (surface) ──────────────────────────────────────────
model_da     = ds_model[MODEL_CHLA_VAR]
mod_time_dim = _find_time_dim(model_da)
mod_z_dim    = _find_vertical_dim(model_da, mod_time_dim)
mod_y_dim, mod_x_dim = _find_xy_dims(model_da, mod_time_dim, mod_z_dim)

if SURFACE_LAYER_CHLA >= model_da.sizes[mod_z_dim] or SURFACE_LAYER_CHLA < -model_da.sizes[mod_z_dim]:
    raise IndexError(
        f"SURFACE_LAYER_CHLA={SURFACE_LAYER_CHLA} out of bounds for "
        f"'{mod_z_dim}' size {model_da.sizes[mod_z_dim]}"
    )
model_surface = model_da.isel({mod_z_dim: SURFACE_LAYER_CHLA})
model_times   = model_surface[mod_time_dim]

lon_name_m = _pick_coord_name(ds_model, ("lonc", "lon", "longitude"))
lat_name_m = _pick_coord_name(ds_model, ("latc", "lat", "latitude"))
if lon_name_m is None or lat_name_m is None:
    raise KeyError("Cannot find lon/lat in model dataset.")

mod_x_map, mod_y_map = _lonlat_to_webmercator(
    _safe_fill_coords(np.asarray(ds_model[lon_name_m].values, dtype=float)),
    _safe_fill_coords(np.asarray(ds_model[lat_name_m].values, dtype=float)),
)

# ── 4. Align times: for each satellite day find the nearest model timestep ────
sat_times_np   = sat_times.values.astype("datetime64[D]")
model_times_np = model_times.values.astype("datetime64[D]")

frame_sat_idx   = np.arange(0, len(sat_times_np), TIME_STEP_COMP)
frame_model_idx = np.array([
    int(np.argmin(np.abs(model_times_np - t))) for t in sat_times_np[frame_sat_idx]
])

print(f"Satellite frames: {len(frame_sat_idx)}, date range: {sat_times_np[0]} – {sat_times_np[-1]}")
print(f"Model time steps matched from: {model_times_np[0]} – {model_times_np[-1]}")

# ── 5. Map extent in Web Mercator ─────────────────────────────────────────────
if MAP_EXTENT_LONLAT is not None:
    _lo = np.array([MAP_EXTENT_LONLAT[0], MAP_EXTENT_LONLAT[1]])
    _la = np.array([MAP_EXTENT_LONLAT[2], MAP_EXTENT_LONLAT[3]])
    _ex, _ey = _lonlat_to_webmercator(_lo, _la)
    xlim_cmp = (float(_ex[0]), float(_ex[1]))
    ylim_cmp = (float(_ey[0]), float(_ey[1]))
else:
    xlim_cmp = (float(np.nanmin(mod_x_map)), float(np.nanmax(mod_x_map)))
    ylim_cmp = (float(np.nanmin(mod_y_map)), float(np.nanmax(mod_y_map)))

# ── 6. Set up figure ──────────────────────────────────────────────────────────
fig_cmp, axes_cmp = plt.subplots(1, 2, figsize=(16, 7), sharey=True)
plt.subplots_adjust(wspace=0.04, right=0.88)
plt.close(fig_cmp)

cbar_ax = fig_cmp.add_axes([0.90, 0.12, 0.018, 0.74])
norm_cmp = plt.Normalize(vmin=CHLA_VMIN, vmax=CHLA_VMAX)
sm_cmp   = plt.cm.ScalarMappable(norm=norm_cmp, cmap=CHLA_CMAP)
sm_cmp.set_array([])
fig_cmp.colorbar(sm_cmp, cax=cbar_ax, label="Chla / CHL (mg m⁻³)")

ax_sat, ax_mod = axes_cmp


def _draw_basemap_cmp(ax):
    if USE_BASEMAP:
        import contextily as ctx
        ctx.add_basemap(
            ax,
            source=BASEMAP_SOURCE,
            crs="EPSG:3857",
            zoom=BASEMAP_ZOOM,
            attribution="",
            zorder=1,
            reset_extent=False,
        )


def update_cmp(frame_number: int):
    si = int(frame_sat_idx[frame_number])
    mi = int(frame_model_idx[frame_number])

    sat_frame = np.asarray(sat_da.isel({sat_time_dim: si}).values, dtype=float)
    if sat_frame.ndim > 2:
        sat_frame = sat_frame.squeeze()

    mod_frame = np.asarray(
        model_surface.isel({mod_time_dim: mi}).transpose(mod_y_dim, mod_x_dim).values,
        dtype=float,
    )

    timestamp_sat = str(sat_times_np[si])
    timestamp_mod = str(model_times_np[mi])

    for ax in (ax_sat, ax_mod):
        ax.clear()
        ax.set_xlim(*xlim_cmp)
        ax.set_ylim(*ylim_cmp)

    _draw_basemap_cmp(ax_sat)
    ax_sat.pcolormesh(
        sat_x_map, sat_y_map, np.ma.masked_invalid(sat_frame),
        shading="auto", cmap=CHLA_CMAP, vmin=CHLA_VMIN, vmax=CHLA_VMAX,
        alpha=SURFACE_ALPHA, zorder=3,
    )
    ax_sat.set_title(f"Satellite CHL | {timestamp_sat}", fontsize=11)
    ax_sat.tick_params(labelbottom=False, labelleft=False)

    _draw_basemap_cmp(ax_mod)
    ax_mod.pcolormesh(
        mod_x_map, mod_y_map, np.ma.masked_invalid(mod_frame),
        shading="auto", cmap=CHLA_CMAP, vmin=CHLA_VMIN, vmax=CHLA_VMAX,
        alpha=SURFACE_ALPHA, zorder=3,
    )
    ax_mod.set_title(f"Model Chla (surface) | {timestamp_mod}", fontsize=11)
    ax_mod.tick_params(labelbottom=False, labelleft=False)

    for ax in (ax_sat, ax_mod):
        ax.set_xlim(*xlim_cmp)
        ax.set_ylim(*ylim_cmp)

    return tuple(ax_sat.collections) + tuple(ax_mod.collections)

plt.rcParams['animation.embed_limit'] = 100  # MB, increase as needed
ani_cmp = FuncAnimation(
    fig_cmp,
    update_cmp,
    frames=len(frame_sat_idx),
    interval=200,
    blit=False,
    repeat=False,
)

cmp_html = HTML(ani_cmp.to_jshtml())

OUTPUT_COMPARISON_GIF.parent.mkdir(parents=True, exist_ok=True)
ani_cmp.save(OUTPUT_COMPARISON_GIF, writer=PillowWriter(fps=5))
print(f"Saved comparison animation to: {OUTPUT_COMPARISON_GIF}")

ds_sat.close()
ds_model.close()

cmp_html

In [ ]:
# Time series trends for all variables in vars_list over the full year 2015
# Comparing spinup_01, spinup_02, spinup_03

SPINUPS = {
    'spinup_01': Path('/export/lv9/projects/dws/model_output/archived_runs/spinup_01/'),
    'spinup_02': Path('/export/lv9/projects/dws/model_output/archived_runs/spinup_02/'),
    'spinup_03': Path('/export/lv9/projects/dws/model_output/archived_runs/spinup_03/'),
    'spinup_04': Path('/export/lv9/projects/dws/model_output/archived_runs/spinup_04/'),
    'spinup_05': Path('/export/lv9/projects/dws/model_output/archived_runs/spinup_05/'),
}
FILE_PATTERN = 'dws_500m.3d.2015??.nc'
SURFACE_LAYER_INDEX = 10
USE_DAILY_MEAN = False
SPINUP_COLORS = ['tab:blue', 'tab:orange', 'tab:green']

def _find_time_dim(da: xr.DataArray) -> str:
    for d in da.dims:
        if 'time' in d.lower():
            return d
    raise ValueError(f'No time dimension found in {da.dims}')

def _find_vertical_dim(da: xr.DataArray, time_dim: str) -> str | None:
    candidates = ('level', 'z', 'sigma', 'layer', 'lev', 'depth', 'nmesh2_layer_3d')
    for d in da.dims:
        if d != time_dim and any(k in d.lower() for k in candidates):
            return d
    return None

def _drop_duplicate_time(da: xr.DataArray, time_dim: str) -> xr.DataArray:
    time_values = np.asarray(da[time_dim].values)
    _, keep_idx = np.unique(time_values, return_index=True)
    keep_idx = np.sort(keep_idx)
    if keep_idx.size < time_values.size:
        da = da.isel({time_dim: keep_idx})
    return da

# ── Load all spinup datasets ──────────────────────────────────────────────────
datasets = {}
for label, data_dir in SPINUPS.items():
    files = sorted(data_dir.glob(FILE_PATTERN))
    if not files:
        print(f'WARNING: No files found for {label} in {data_dir} — skipping.')
        continue
    datasets[label] = xr.open_mfdataset(
        files,
        combine='nested',
        concat_dim='time',
        decode_times=True,
        data_vars='minimal',
        coords='minimal',
        compat='override',
        join='override',
    )
    print(f'Loaded {label}: {len(files)} files')

if not datasets:
    raise FileNotFoundError('No spinup datasets could be loaded.')

# ── Plot ──────────────────────────────────────────────────────────────────────
nvars = len(vars_list)
ncols = 4
nrows = int(np.ceil(nvars / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 2.8 * nrows), sharex=True)
axes = np.atleast_1d(axes).ravel()

missing_vars = []
failed_vars  = []

for i, vname in enumerate(vars_list):
    ax = axes[i]
    plotted = False

    for (label, ds), color in zip(datasets.items(), SPINUP_COLORS):
        try:
            if vname not in ds.variables:
                continue

            da = ds[vname].squeeze(drop=True)
            time_dim = _find_time_dim(da)
            z_dim    = _find_vertical_dim(da, time_dim)

            da_use = da.isel({z_dim: SURFACE_LAYER_INDEX}) if z_dim is not None else da
            da_use = _drop_duplicate_time(da_use, time_dim)

            spatial_dims = [d for d in da_use.dims if d != time_dim]
            if not spatial_dims:
                raise ValueError('No spatial dimensions left for domain averaging.')

            if vname.lower() in ('chla', 'netppm2'):
                da_use = da_use.where(da_use >= 0)

            series = da_use.mean(dim=spatial_dims, skipna=True)

            if USE_DAILY_MEAN:
                series = series.resample({time_dim: '1D'}).mean(skipna=True)

            ax.plot(series[time_dim].values, series.values, lw=1.6, color=color, label=label)
            plotted = True

        except Exception as e:
            failed_vars.append((vname, label, str(e)))

    if not plotted:
        missing_vars.append(vname)
        ax.text(0.5, 0.5, f'{vname}\nnot found', ha='center', va='center', transform=ax.transAxes)
        ax.set_axis_off()

    units = ''
    for ds in datasets.values():
        if vname in ds.variables:
            units = ds[vname].attrs.get('units', '')
            break
    ax.set_title(vname)
    ax.set_ylabel(f'{vname} [{units}]' if units else vname, fontsize=8)
    ax.grid(True, alpha=0.3)

for j in range(nvars, len(axes)):
    axes[j].set_visible(False)

for ax in axes[:nvars]:
    ax.tick_params(axis='x', labelrotation=35)

# Single shared legend in the first empty panel or as a figure legend
handles = [plt.Line2D([0], [0], color=c, lw=1.6, label=l)
           for l, c in zip(datasets.keys(), SPINUP_COLORS)]
fig.legend(handles=handles, loc='lower right', fontsize=10, framealpha=0.9)

fig.suptitle('2015 Domain-Mean Trends — spinup comparison', y=1.02)
plt.tight_layout()
plt.show()

# Close datasets
for ds in datasets.values():
    ds.close()

if missing_vars:
    print('Variables not found in any dataset:', missing_vars)
if failed_vars:
    print('Variables that failed during processing:')
    for name, label, err in failed_vars:
        print(f'  - {name} ({label}): {err}')